In [1]:
import geopandas as gpd
import geopandas as gpd
from shapely.ops import unary_union, polygonize
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress

In [2]:
points = gpd.read_file("/Volumes/MaloneLab/Research/Whitezone/Result/Spatial_Modelling/Results/combined_hydro.shp")
# Ensure CRS is geographic (lat/lon), if not reproject to WGS84
if points.crs != "EPSG:4326":
    points = points.to_crs("EPSG:4326")

# Extract lat/lon
points["lon"] = points.geometry.x
points["lat"] = points.geometry.y

# Now you have a DataFrame with lat/lon
print(points[["lat", "lon"]].head())

# 2. Read transects shapefile
transects = gpd.read_file("/Volumes/MaloneLab/Research/Whitezone/Spatial_Data/transects_39_61.shp")

# Reproject if needed to match CRS
if transects.crs != "EPSG:4326":
    transects = transects.to_crs("EPSG:4326")

print(transects.head())

         lat        lon
0  25.175889 -80.735583
1  25.175556 -80.792778
2  25.224566 -80.540061
3  25.232625 -80.524867
4  25.198611 -80.618879
   ID                                           geometry
0  39  LINESTRING (-80.43141 25.24549, -80.47312 25.3...
1  61  LINESTRING (-80.51986 25.22790, -80.55485 25.2...


In [3]:
west = [
    "NP-P37", 
    "NP-146", 
    "NP-OL", 
    "Mud_creek_at_mouth", 
    "SWEVER5A",
    "NP-CHP",    
]

central = [
    "EVER7", 
    "CT50R", 
    "EVER6", 
    "NP-EPSW", 
    "EPSW", 
    "CT27R", 
    "NP-EP1", 
    "EVER8",
    "HIGHWAY_CR",
    "ENPLS",
    
]

east = [
     "SWEVER1",
     "S197_H",
     "S197_T",
     "S20_T",
     "S20_H"
]

In [4]:
### read EDEN data ##
hydro_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Hydro/EDEN/annual_summary.csv')

# Keep only specified sites
keep_sites = ['CT27R', 'CT50R', 'EPSW', 'EVER6', 'EVER7', 'EVER8', 'G-1251', 'Mud_creek_at_mouth']
hydro_df = hydro_df[hydro_df['source_file'].isin(keep_sites)]


# Convert types
hydro_df['year'] = hydro_df['year'].astype(int)
hydro_df = hydro_df[hydro_df['year'] != 2025]
hydro_df['mean_value_NAVD88'] = pd.to_numeric(hydro_df['mean_value'], errors='coerce') * 30.48

In [5]:
dbhydro_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Hydro/DBhydro/annual_summary.csv')

# Read the site metadata with datum info
datum_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Hydro/DBhydro/WZ_hydro_sites_latlon.csv')

dbhydro_df = dbhydro_df.merge(
    datum_df[['Name', 'Datum', 'Conversion']],
    left_on='source_file',
    right_on='Name',
    how='left'
)

# Drop the redundant 'Name' column from the merge
dbhydro_df.drop(columns=['Name'], inplace=True)

# Preview result
print(dbhydro_df.head())

# Convert mean_value to NAVD88 only when Datum is '29'
dbhydro_df['mean_value_NAVD88'] = dbhydro_df.apply(
    lambda row: row['mean_value'] + row['Conversion'] if str(row['Datum']).strip() == '29' else row['mean_value'],
    axis=1
)

# Preview the result

dbhydro_df['mean_value_NAVD88'] = dbhydro_df['mean_value_NAVD88'] * 30.48

combined_df = pd.concat([dbhydro_df, hydro_df], join='inner', ignore_index=True)

   year  mean_value  sd_value source_file  lat_decimal  lon_decimal  Datum  \
0  1996   -0.815611  0.321434       ENPGB     25.16723     -80.8013   88.0   
1  1997   -0.643000  0.358060       ENPGB     25.16723     -80.8013   88.0   
2  1998   -0.645975  0.333910       ENPGB     25.16723     -80.8013   88.0   
3  1999   -0.551192  0.380846       ENPGB     25.16723     -80.8013   88.0   
4  2000   -0.628565  0.401476       ENPGB     25.16723     -80.8013   88.0   

   Conversion  
0         NaN  
1         NaN  
2         NaN  
3         NaN  
4         NaN  


In [6]:
combined_df

,year,mean_value,sd_value,source_file,lat_decimal,lon_decimal,mean_value_NAVD88
0,1996,-0.815611,0.321434,ENPGB,25.16723,-80.8013,-24.859827
1,1997,-0.643000,0.358060,ENPGB,25.16723,-80.8013,-19.598640
2,1998,-0.645975,0.333910,ENPGB,25.16723,-80.8013,-19.689328
3,1999,-0.551192,0.380846,ENPGB,25.16723,-80.8013,-16.800334
4,2000,-0.628565,0.401476,ENPGB,25.16723,-80.8013,-19.158648
...,...,...,...,...,...,...,...
1218,2020,-0.231421,0.399717,Mud_creek_at_mouth,NaN,NaN,-7.053705
1219,2021,-0.258164,0.320264,Mud_creek_at_mouth,NaN,NaN,-7.868850
1220,2022,-0.087644,0.385247,Mud_creek_at_mouth,NaN,NaN,-2.671384
1221,2023,-0.062521,0.368535,Mud_creek_at_mouth,NaN,NaN,-1.905626


In [12]:
## Analyze the trend between 2013 to 2024
# Filter years 2013–2024
test_df = combined_df[(combined_df["year"] >= 2014) & (combined_df["year"] <= 2024)].copy()

# Assign region
def assign_region(station):
    if station in west:
        return "west"
    elif station in central:
        return "central"
    elif station in east:
        return "east"
    else:
        return None  # ignore stations not listed

test_df["region"] = test_df["source_file"].apply(assign_region)

# Subtract station mean instead of 2013 baseline
demeaned_list = []
for station, subdf in test_df.groupby("source_file"):
    subdf = subdf.copy()
    mean_val = subdf["mean_value_NAVD88"].mean()
    subdf["demeaned"] = subdf["mean_value_NAVD88"] - mean_val
    demeaned_list.append(subdf)

demeaned_df = pd.concat(demeaned_list)

# Calculate slope per region using demeaned values
trends = {}
for region, subdf in demeaned_df.groupby("region"):
    # Aggregate by year: mean across stations in the region
    yearly_mean = subdf.groupby("year")["demeaned"].mean().reset_index()
    
    slope, intercept, r_value, p_value, std_err = linregress(yearly_mean["year"], yearly_mean["demeaned"])
    
    trends[region] = {
        "slope": slope,
        "p_value": p_value,
        "r2": r_value**2
    }

trend_df = pd.DataFrame(trends).T
print("Demeaned regression trends (2013–2024):")
print(trend_df)


Demeaned regression trends (2013–2024):
            slope   p_value        r2
central  1.156313  0.000875  0.725503
east     1.064582  0.001689  0.683889
west     1.059934  0.000662  0.741490


In [10]:
# Count original rows
original_rows = combined_df.shape[0]

# Remove duplicates: keep first occurrence per station per year
combined_df = combined_df.drop_duplicates(subset=["source_file", "year"])

# Count rows after deduplication
new_rows = combined_df.shape[0]

# Report how many rows were removed
rows_removed = original_rows - new_rows
print(f"Removed {rows_removed} duplicate rows.")

Removed 141 duplicate rows.


In [11]:
%matplotlib agg
### Plot with mean values ####
plt.rcParams["font.family"] = "Times New Roman"
fs = 16

# Filter years 1996–2024
df = combined_df[(combined_df["year"] >= 1996) & (combined_df["year"] <= 2024)]

# Filter only the stations in each region
west_df = df[df["source_file"].isin(west)]
central_df = df[df["source_file"].isin(central)]
east_df = df[df["source_file"].isin(east)]

# Restrict East also to 1996–2024
east_df = east_df[(east_df["year"] >= 1996) & (east_df["year"] <= 2024)]

plt.figure(figsize=(12, 6))

# Define regions and colors
regions = {
    "West": (west_df, "#66c2a5"),
    "Central": (central_df, "#fc8d62"),
    "East": (east_df, "#8da0cb")
}

# Plot each region
for name, (subdf, color) in regions.items():
    if subdf.empty:
        continue  # skip if no data
    
    # Grey station lines
    for station, sdf in subdf.groupby("source_file"):
        plt.plot(sdf["year"], sdf["mean_value_NAVD88"], 
                 color="grey", alpha=0.4, linewidth=1, label="_nolegend_")
    
    # Compute yearly stats
    grouped = subdf.groupby("year")["mean_value_NAVD88"]
    mean_vals = grouped.mean()
    min_vals = grouped.min()
    max_vals = grouped.max()
    
    # Shaded region (no hatch)
    plt.fill_between(mean_vals.index, min_vals, max_vals,
                     color=color, alpha=0.12, edgecolor=color, linewidth=0.5)
    
    # Mean line
    plt.plot(mean_vals.index, mean_vals.values, 
             color=color, linewidth=2, label=name)

# Labels and title
plt.xlabel("Year", fontsize=fs)
plt.ylabel("Water Level (cm)", fontsize=fs)
plt.title("Annual Mean Water Level", fontsize=fs)
plt.legend(fontsize=fs, bbox_to_anchor=(1.05, 1), loc="upper left")

# Custom ticks from 1996 → 2024
years = list(range(1996, 2025, 5))
if years[-1] != 2024:
    years.append(2024)

plt.xticks(ticks=years, labels=years, fontsize=fs)
plt.yticks(fontsize=fs)

# Lock axis range explicitly
plt.xlim(1996, 2024)

plt.grid(alpha=0.3)
plt.tight_layout()

# Save figure first, then show
plt.savefig("/Users/bz294/Documents/WhiteZone/Figure/Backup/wl_plot.tif", dpi=300, bbox_inches="tight")
plt.show()


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_94267/4004257786.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
def check_stations_exist(df, station_list):
    source_files = df['source_file'].unique()
    missing = [s for s in station_list if s not in source_files]
    if missing:
        print(f"Missing stations: {missing}")
    else:
        print("All stations are present.")

# Check each group
print("West stations:")
check_stations_exist(combined_df, west)

print("\nCentral stations:")
check_stations_exist(combined_df, central)

print("\nEast stations:")
check_stations_exist(combined_df, east)

In [ ]:
# Combine all stations into a single set
all_stations = set(west + central + east)

# Find any source_file not in the combined lists
extra_stations = combined_df.loc[~combined_df['source_file'].isin(all_stations), 'source_file'].unique()

if len(extra_stations) > 0:
    print("Source files not in west, central, or east lists:")
    print(extra_stations)
else:
    print("All source_file values are included in the lists.")

In [ ]:
import pandas as pd

# Combine all station lists
all_stations = west + central + east

# Filter DataFrame by source_file names
df_filtered = combined_df[combined_df['source_file'].isin(all_stations)]




In [ ]:
import pandas as pd

# Define the time range
start_year = 1996
end_year = 2024

# Filter df_filtered for years 1996-2024
df_timeseries = df_filtered[(df_filtered['year'] >= start_year) & (df_filtered['year'] <= end_year)]

# Find which stations have any data in this period
stations_with_timeseries = df_timeseries['source_file'].unique()
print("Stations with time series between 1996-2024:")
print(stations_with_timeseries)

all_stations = west + central + east

# Find stations from your list that do NOT have data in 1996-2024
missing_stations = [s for s in all_stations if s not in stations_with_timeseries]

if missing_stations:
    print("Stations from the list missing data in 1996-2024:")
    print(missing_stations)
else:
    print("All stations in the list have data in 1996-2024.")

